## Apresentação ✒️

Notebook destinado a realizar a interação com o modelo de Conversational RAG que será avaliado, posteriormente com base nas métricas baseadas em similaridade vetorial e pela abordagem conhecida como LLM as a Judge. Aqui constará três chatbots : um que responde ao usuário com base na sua base de conhecimento e outros dois que farão a mesma coisa, sendo, no entanto, a eles apresentados mensagens que visam produzir falhas lógicas em sua resposta - para esses, será avaliado a consistência do modelo e a alternativa proposta, visando conceber como tais aspectos podem ser arrefecidos em um contexto de Conversational RAG. 

Para fins de contextualização, Conversational RAG é o nome que se dá para aplicações que utilizam de modelos generativos - NLG - para servirem como chatbots atrelados a determinados contextos, para os quais contar com o pré-treinamento do modelo generativo utilizado não necessariamente irá produzir êxito, necessitando que a ele sejam fornecidas bases de conhecimento a partir das quais auxiliam no processo de geração de resposta mais acurada. 

### Library 📓

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [12]:
import os
import getpass
import evaluate
import numpy as np
import pandas as pd
import logging

from tqdm import tqdm

from typing import Dict, List

from IPython.display import Markdown

from features.clean_memory import CleanMemory

from prompts.system_prompt_template import system_prompt_template

from pandas import DataFrame

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_community.document_loaders import PyPDFLoader

from langchain_core.chat_history import (BaseChatMessageHistory,
                                         InMemoryChatMessageHistory)
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import HumanMessage, trim_messages
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import (ChatPromptTemplate, MessagesPlaceholder,
                                    PromptTemplate)
from langchain_core.runnables import Runnable, RunnableBranch, RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import PydanticOutputParser, JsonOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_core.vectorstores import InMemoryVectorStore, VectorStore

from langchain_groq import ChatGroq

from langchain_huggingface.embeddings import HuggingFaceEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_google_genai import ChatGoogleGenerativeAI

from pydantic import BaseModel

from sentence_transformers import SentenceTransformer
from sentence_transformers.util import pairwise_cos_sim

### Inicializando o modelo de LLM

In [3]:
# API reference : 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

os.environ["GROQ_API_KEY"]=getpass.getpass("Your API Key: ")

In [ ]:
qwen_qwen  = "qwen/qwen3-32b"
mini_llama = "llama3-8b-8192"
llama_2    = "llama3-70b-8192"
llama      = "llama-3.3-70b-versatile"
deepseek   = "deepseek-r1-distill-llama-70b"

llm = ChatGroq(
    model = llama_2, 
    temperature = 0
)   

llm.invoke("Olá, tudo bem ?").content

'Olá! Tudo bem, obrigado! E você?'

### Embedding

In [6]:
%%time

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

CPU times: total: 2.27 s
Wall time: 6.5 s


### Formando a base de conhecimento 

A base de conhecimento utilizada se refere à lore do anime/ mangá Tokyo Ghoul - uma das melhores obras góticas - a partir da qual o modelo deverá utilizar para responder a certas perguntas do usuário, também sobre o tema. 

In [7]:
%%time

"""
Elaborando os métodos utilizados para o modelo possuir
a sua base de conhecimento.  
"""

loader = PyPDFLoader("./data/Tokyo Ghoul Knowledge Base.pdf").load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size         = 500, 
    chunk_overlap      = 50, 
    length_function    = len,
    separators         = ["", " ", ".", "\n", "\n\n"],
    is_separator_regex = False
).split_documents(loader)    

retriever = InMemoryVectorStore.from_documents( 
    documents = text_splitter,
    embedding = embeddings
).as_retriever(search_kwargs={"k": 2})

CPU times: total: 35.4 s
Wall time: 23.4 s


In [29]:
kb_recovered = retriever.invoke("Psi")

In [30]:
content = [doc.page_content for doc in kb_recovered]
content

['típica dos gêneros de terror, contrasta com cenas \nde introspecção e sofrimento psicológico, remetendo a obras que exploram a condição \nhumana diante de forças incontroláveis. Essa combinação reforça uma atmosfera',
 'cções de ghouls, \ndesencadeando alianças improváveis e confrontos de grandes proporções. \n \n2. Cenário']

### Dataset utilizado

In [8]:
file_name = "eval_dataset_tokyo_ghoul.xlsx"

df = pd.read_excel(f"./data/{file_name}", engine="openpyxl")

In [9]:
df = df.drop("Unnamed: 3", axis=1)

In [10]:
df.head()

,Question,Ground Truth,Base de conhecimento
0,O que são ghouls em Tokyo Ghoul e como eles se...,Os ghouls são criaturas muito semelhantes aos ...,Os ghouls são seres são fisicamente muito seme...
1,Como e por que foi criada a organização CCG?,À medida que cresciam os conflitos e as mortes...,"Originalmente, a sociedade humana desconhece a..."
2,O que acontece com Ken Kaneki após o transplan...,"Ken Kaneki, um estudante universitário, sofre ...",Esse procedimento transforma Kaneki em um meio...
3,Quais diferentes visões de convivência entre g...,Existem facções que defendem a paz e a coexist...,Alguns grupos de ghouls defendem a paz e tenta...
4,: Qual é o significado de “One-Eyed King” no u...,O “One-Eyed King” (Rei de Olho Único) é uma fi...,Espécime de figura messiânica para alguns ghou...


### GhoulBot 🦇

Chatbot utilizado para responder às perguntas presentes no dataset informado, com o objetivo de gerar as respostas do modelo que depois serão verificadas pelo Judge. 

In [11]:
# Configura o logging para exibir mensagens no console
logging.basicConfig(
    level=logging.DEBUG,  # Mostra logs DEBUG, INFO, WARNING, ERROR e CRITICAL
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

In [ ]:
class GhoulBot:
    """
    Conversational RAG (Retrieval-Augmented Generation) system for handling 
    conversational queries with context-aware retrieval.
    """
    def __init__(
            self, 
            llm: BaseChatModel, 
            system_prompt: str,
            retriever: VectorStore, 
        ) -> None:
        """
        Initializes the ConversationalRag instance.

        Args:
            llm (BaseChatModel): The language model used for response generation.
            system_message (str): The system-level instruction message.
            contextualize_message (str): The message to provide context-aware queries.
            embedding (Embeddings): The embedding model used for document retrieval.
            memory (ChatMessageHistory): The chat history manager.
            documents (List[Document]): A list of documents to be processed and retrieved.
        """
        self.llm           = llm 
        self.system_prompt = system_prompt
        self.retriever     = retriever

    def log_documents(self, query: str) -> str:
        """
        Log documents in order to query provided. 
        Args:
            query: The user's message query.  
        """
        kb_recovered = self.retriever.invoke(query)
        content = [doc.page_content for doc in kb_recovered]
        return content
        
    def run(self, query: str) -> str:
        """ 
        Executes the RAG pipeline for a given query and returns the generated response.

        Args:
            query (str): The user input query.
        
        Returns:
            str: The generated response from the conversational model.
        """ 
        rag_chain = (
            {"context": self.retriever, "question": RunnablePassthrough()}
            | self.system_prompt
            | self.llm
            | StrOutputParser()
        )
        
        response = rag_chain.invoke(query)
        kb_recovered = self.log_documents(query=query)

        return {
            "response_model": response, 
            "kb_recovered": kb_recovered
        }

In [ ]:
# Incializando o bot:

ghoul_bot = GhoulBot(
    llm           = llm, 
    system_prompt = system_prompt_template, 
    retriever     = retriever
)

### Interagindo com o modelo

In [33]:
""" 
Verificando a pergunta presente no dataset. 
"""

user_message = df["Question"][5]
print(f"User message: {user_message}")

User message: De que forma Ken Kaneki se envolve com diferentes facções ao longo da história?


In [ ]:
%%time

# Verificando a resposta do modelo e as bases
# de conhecimento recuperadas. 

user_message = df["Question"][5]

response = ghoul_bot.run(query=user_message)

response

CPU times: total: 2.53 s
Wall time: 4.58 s


{'response': 'Ken Kaneki se envolve com diferentes facções ao longo da história de Tokyo Ghoul devido à sua condição de meio-ghoul e sua busca por sobrevivência e aceitação. Inicialmente, ele se aproxima do Anteiku, um café gerenciado por ghouls que se esforçam para coexistir pacificamente com humanos. Lá, ele encontra apoio e amizade de personagens como Touka Kirishima e Yomo Renji. No entanto, após ser torturado por Yamori, Kaneki se torna mais agressivo e começa a se aproximar da facção dos ghouls radicais, liderados por Ayato Kirishima, que buscam lutar contra a Comissão de Controle de Ghouls (CCG) e defender os direitos dos ghouls. Mais tarde, em Tokyo Ghoul:re, Kaneki assume a identidade de Haise Sasaki e trabalha para a CCG, liderando a equipe Quinx, composta por meio-ghouls treinados para caçar ghouls. Nessa posição, ele se envolve em uma luta interna entre a CCG e os ghouls, enquanto também luta para manter sua sanidade e valores morais.',
 'kb_recovered': ['nas do mangá, \nin

In [27]:
"""
Devido a API utilizada não lidar bem com muitas requisições, irei quebrar 
dataset em dois - cabeçalho e rodapé, condiderando as 15 primeiras e últimas
perguntas para cada `run` do modelo.
"""

df_head = df.head(15)
df_tail = df.tail(15)

In [ ]:
# Armazenando os dois ciclos de iteração em listas 
# diferentes para que depois possam ser 'mergeadas'. 

""" 
responses_1 = []
Kbs_recovered_1 = []

responses_2 = []
Kbs_recovered_2 = []
"""
responses = []
Kbs_recovered = []


for i in tqdm(range(30), desc="Gerando a resposta"):
    question = df["Question"][i]       
    
    output = ghoul_bot.run(question)

    response = output["response_model"]
    Kb_recovered = output["kb_recovered"]

    responses.append(response)
    Kbs_recovered.append(Kb_recovered)

Gerando a resposta: 100%|██████████| 30/30 [07:05<00:00, 14.18s/it]


In [44]:
# Verificando as respostas trazidas pelo modelo.

responses[:5]

['Os ghouls são criaturas muito semelhantes aos humanos, mas possuem órgãos internos chamados “RC cells” que os obrigem a se alimentar de carne humana para sobreviver. Além disso, cada ghoul desenvolve um apêndice predatório denominado “kagune”, que lhe confere habilidades de combate sobrenaturais.',
 'A organização CCG (Comissão de Contra-Ghoul) foi criada como uma resposta à crescente ameaça representada pelos ghouls na sociedade humana. À medida que os conflitos e as mortes misteriosas aumentavam, a necessidade de uma força policial secreta para investigar e combater essas criaturas tornou-se cada vez mais urgente. O CCG foi estabelecido para recrutar e treinar Investigadores de Ghoul, humanos que dedicam suas vidas a caçar e neutralizar ghouls, utilizando quinques — armas feitas a partir do kagune dos ghouls. A criação do CCG marca um esforço para proteger a humanidade da ameaça ghoul e manter a ordem na sociedade.',
 'Ken Kaneki, um estudante universitário, sofre um grave acidente

In [ ]:
# Verificando as bases de conhecimento trazidas pelo modelo.

Kbs_recovered[:5]

[['inhas entre \nbem e mal tornam-se tênues. \nAlém disso, a série introduz conceitos como o “One-Eyed King” (Rei de Olho Único), \nfigura messiânica para alguns ghouls, que simboliza esperança, poder revolucionário e a \npossibilidade de um futuro em que ghouls e humanos alcancem um equilíbrio. Esse \ntítulo é cercado de lendas, ambições e conspirações. Ao longo das sagas “Tokyo Ghoul” \ne “Tokyo Ghoul:re”, essa figura influencia tanto o CCG quanto as facções de ghouls, \ndesencadeando alianças improváve',
  'medida que ghouls e humanos \nse enfrentam em batalha. Essa ambientação contribui para a construção de um universo \nonde a segurança é efêmera, e qualquer um pode se tornar caçador ou presa a qualquer \nmomento. \n \n3. Influência Cultural e Estética \nA estética de Tokyo Ghoul combina elementos de horror corporal (body horror) com \ninfluências do mangá e anime seinen, voltados para um público mais maduro. A \nviolência visceral, impiedosa e gráfica, típica dos gêneros de terro

In [45]:
df["Response Model"] = responses
df["Kbs Recovered"] = Kbs_recovered

In [46]:
df.head()

,Question,Ground Truth,Base de conhecimento,Response Model,Kbs Recovered
0,O que são ghouls em Tokyo Ghoul e como eles se...,Os ghouls são criaturas muito semelhantes aos ...,Os ghouls são seres são fisicamente muito seme...,Os ghouls são criaturas muito semelhantes aos ...,[inhas entre \nbem e mal tornam-se tênues. \nA...
1,Como e por que foi criada a organização CCG?,À medida que cresciam os conflitos e as mortes...,"Originalmente, a sociedade humana desconhece a...",A organização CCG (Comissão de Contra-Ghoul) f...,[ca por um meio-\ntermo entre a sobrevivência ...
2,O que acontece com Ken Kaneki após o transplan...,"Ken Kaneki, um estudante universitário, sofre ...",Esse procedimento transforma Kaneki em um meio...,"Ken Kaneki, um estudante universitário, sofre ...","[nas do mangá, \ninfluenciando tendências esté..."
3,Quais diferentes visões de convivência entre g...,Existem facções que defendem a paz e a coexist...,Alguns grupos de ghouls defendem a paz e tenta...,"Em Tokyo Ghoul, existem diferentes visões de c...",[entre humanos e ghouls. Alguns \ngrupos de gh...
4,: Qual é o significado de “One-Eyed King” no u...,O “One-Eyed King” (Rei de Olho Único) é uma fi...,Espécime de figura messiânica para alguns ghou...,"No universo de Tokyo Ghoul, o ""One-Eyed King"" ...",[inhas entre \nbem e mal tornam-se tênues. \nA...


In [49]:
file_name = "dataset_response_model"
df.to_excel(f"{file_name}.xlsx")